[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/exercises/intermediate/03_debugging_ml_code.ipynb)

# Luyện tập trung cấp 3 — Gỡ lỗi code ML

Mỗi bài có một đoạn code **chạy được nhưng sai** — kiểu lỗi phổ biến nhất trong ML vì không có thông báo lỗi nào. Đọc, tìm lỗi, sửa trong khối `START/END`. Ghi lại cho mình: lỗi thuộc loại nào, làm sao phát hiện sớm hơn.

Chương liên quan: 01, 04, 05, 14, 15, 17, 19.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/exercises/intermediate

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.utils import repo_path

rng = np.random.default_rng(0)

**Bài 1 — Chuẩn hoá sai trục.** Hàm dưới cần chuẩn hoá mỗi **đặc trưng** (cột) về trung bình 0, độ lệch chuẩn 1.

In [ ]:
def standardize(X):
    ### START CODE HERE ### (≈ 1 dòng)
    return (X - X.mean(axis=1, keepdims=True)) / X.std(axis=1, keepdims=True)
    ### END CODE HERE ###


X1 = rng.normal([10, 200, -5], [1, 50, 3], (500, 3))

In [ ]:
check_close(standardize(X1).mean(axis=0), np.zeros(3), tol=1e-9, message="mỗi cột trung bình 0")
check_close(standardize(X1).std(axis=0), np.ones(3), message="mỗi cột độ lệch chuẩn 1")

**Bài 2 — Rò rỉ qua `fit_transform` trên test.**

In [ ]:
X2, y2 = rng.normal(size=(1000, 4)), rng.integers(0, 2, 1000)
X2_tr, X2_te = X2[:800], X2[800:] * 3 + 5                               # test có thang đo khác (vd đơn vị khác)


def scale_train_test(X_train, X_test):
    ### START CODE HERE ### (≈ 2 dòng)
    scaler = StandardScaler()
    return scaler.fit_transform(X_train), scaler.fit_transform(X_test)
    ### END CODE HERE ###


tr_s, te_s = scale_train_test(X2_tr, X2_te)

In [ ]:
check_close(te_s.mean(axis=0), (X2_te.mean(axis=0) - X2_tr.mean(axis=0)) / X2_tr.std(axis=0), message="test dùng thống kê của train")
check_greater(np.abs(te_s.mean(axis=0)).min(), 1, "khác biệt thang đo của test vẫn lộ ra (không bị 'chuẩn hoá mất')")

**Bài 3 — Chia theo thời gian lệch một ngày.** Hàm cần trả về train = mọi ngày **trước** `cutoff`, test = từ `cutoff` trở đi, không trùng.

In [ ]:
days = pd.DataFrame({"date": pd.date_range("2024-01-01", periods=100), "value": rng.normal(size=100)})


def time_split(df, cutoff):
    ### START CODE HERE ### (≈ 2 dòng)
    return df[df.date <= cutoff], df[df.date >= cutoff]
    ### END CODE HERE ###


tr3, te3 = time_split(days, "2024-03-01")

In [ ]:
check_equal(len(tr3) + len(te3), 100, "không trùng, không mất ngày")
check_true(tr3.date.max() < te3.date.min(), "train hoàn toàn trước test")

**Bài 4 — F1 sai cách tính trung bình.** Dữ liệu 3 lớp mất cân bằng; báo cáo yêu cầu F1 **trung bình đều các lớp** (mỗi lớp quan trọng như nhau).

In [ ]:
y4_true = np.r_[np.zeros(900), np.ones(80), np.full(20, 2)].astype(int)
y4_pred = y4_true.copy()
y4_pred[900:940] = 0                                                   # nửa lớp 1 bị đoán nhầm thành lớp 0
y4_pred[980:] = 0                                                      # toàn bộ lớp 2 bị bỏ sót


def report_f1(y_true, y_pred):
    ### START CODE HERE ### (≈ 1 dòng)
    return f1_score(y_true, y_pred, average="micro")
    ### END CODE HERE ###

In [ ]:
check_less(report_f1(y4_true, y4_pred), 0.6, "macro-F1 phản ánh việc bỏ sót hoàn toàn lớp hiếm")
check_greater(f1_score(y4_true, y4_pred, average="micro"), 0.9, "(micro-F1 = accuracy — che giấu vấn đề)")

**Bài 5 — Cột số bị đọc thành chuỗi.** File CSV ghi tiền bằng dấu chấm phân cách nghìn ("1.250.000"); code tính trung bình sai.

In [ ]:
raw5 = pd.DataFrame({"amount": ["1.250.000", "980.000", "2.000.000", "15.500.000"]})


def mean_amount(df):
    ### START CODE HERE ### (≈ 1 dòng)
    return df.amount.str.replace(",", "").astype(float).mean()
    ### END CODE HERE ###

In [ ]:
check_close(mean_amount(raw5), (1_250_000 + 980_000 + 2_000_000 + 15_500_000) / 4, message="đọc đúng số tiền")

**Bài 6 — Kết quả không tái lập.** Hàm đánh giá cho ra số khác nhau mỗi lần chạy. Sửa để **hai lần gọi giống hệt nhau** (seed cho chia dữ liệu và mô hình).

In [ ]:
churn = pd.read_csv(repo_path("data", "sample", "customer_churn.csv"))
X6 = churn[["tenure_months", "monthly_charges", "support_calls"]]
y6 = churn.churn


def evaluate_once():
    ### START CODE HERE ### (≈ 3 dòng)
    X_tr, X_te, y_tr, y_te = train_test_split(X6, y6, test_size=0.3)
    model = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_tr, y_tr)
    return model.score(X_te, y_te)
    ### END CODE HERE ###

In [ ]:
check_equal(evaluate_once(), evaluate_once(), "hai lần chạy cho cùng kết quả")

**Bài 7 — Chọn đặc trưng ngoài cross-validation.** Dữ liệu 100 mẫu, 2.000 đặc trưng **nhiễu thuần**. Code gốc chọn 20 đặc trưng tương quan nhất với nhãn trên **toàn bộ dữ liệu** rồi mới CV → AUC rất cao. Sửa bằng cách đưa bước chọn vào pipeline (`SelectKBest(f_classif, k=20)`) → `honest_auc`.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif  # noqa: E402

X7, y7 = rng.normal(size=(100, 2000)), rng.integers(0, 2, 100)
selected = SelectKBest(f_classif, k=20).fit(X7, y7).get_support()
leaky_auc = cross_val_score(LogisticRegression(max_iter=1000), X7[:, selected], y7, cv=5, scoring="roc_auc").mean()
### START CODE HERE ### (≈ 2 dòng)
pipeline = None
honest_auc = None
### END CODE HERE ###
print(f"chọn ngoài CV: {leaky_auc:.3f} | chọn trong CV: {honest_auc:.3f} (sự thật: 0.5)")

In [ ]:
check_greater(leaky_auc, 0.85, "tái hiện: chọn trên toàn bộ dữ liệu cho AUC ảo")
check_less(abs(honest_auc - 0.5), 0.15, "chọn trong pipeline → gần sự thật")